![python_db_connection_cursor_flow.png](attachment:python_db_connection_cursor_flow.png)


In [ ]:
import sqlite3
connection = sqlite3.connect("Online_Course")
cursor = connection.cursor()


### **Create Tables**
#### **Table 1: Staff**


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Staff 
(
    StaffID INT PRIMARY KEY,
    StaffName VARCHAR(100) NOT NULL,
    Department VARCHAR(50) NOT NULL,
    MonthlySalary DECIMAL(10,2) NOT NULL,
    HireDate DATE NOT NULL
)
""")


#### **Table 2: Events**


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Events
(
    EventID INT PRIMARY KEY,
    EventName VARCHAR(100) NOT NULL,
    ClientName VARCHAR(100) NOT NULL,
    EventType VARCHAR(50) NOT NULL,
    Budget DECIMAL(12,2) NOT NULL,
    EventDate DATE NOT NULL
)
""")


#### **Table 3: EventAssignments**


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS EventAssignments
(
    AssignmentID INT PRIMARY KEY,
    StaffID INT NOT NULL,
    EventID INT NOT NULL,
    Role VARCHAR(50) NOT NULL,
    HoursWorked INT NOT NULL,
    HourlyRate DECIMAL(8,2) NOT NULL,

    FOREIGN KEY (StaffID) REFERENCES Staff(StaffID),
    FOREIGN KEY (EventID) REFERENCES Events(EventID)
)
""")


### **Data Insertions**
#### **Staff**


In [ ]:
cursor.execute("""
INSERT INTO Staff
(StaffID, StaffName, Department, MonthlySalary, HireDate)
VALUES
(1, 'Ahmed Samir', 'Operations', 18000, '2022-03-10'),
(2, 'Mona Ali', 'Marketing', 22000, '2021-06-15'),
(3, 'Omar Hassan', 'Technical', 26000, '2020-02-20'),
(4, 'Sara Mohamed', 'Operations', 21000, '2023-01-12'),
(5, 'Youssef Adel', 'Technical', 24000, '2021-09-05'),
(6, 'Nour Khaled', 'Marketing', 19500, '2022-11-18'),
(7, 'Mahmoud Tarek', 'Technical', 28500, '2019-05-25'),
(8, 'Salma Mostafa', 'Operations', 23000, '2020-10-14'),
(9, 'Karim Hany', 'Marketing', 17500, '2024-01-10'),
(10, 'Laila Ahmed', 'Operations', 20500, '2023-04-22')
""")


#### **Events**


In [ ]:
cursor.execute("""
INSERT INTO Events
(EventID, EventName, ClientName, EventType, Budget, EventDate)
VALUES
(101, 'AI Innovation Summit', 'TechVision', 'Conference', 600000, '2026-10-15'),
(102, 'Annual Banking Forum', 'Future Bank', 'Conference', 750000, '2026-11-05'),
(103, 'Digital Marketing Workshop', 'Smart Media', 'Workshop', 250000, '2026-10-22'),
(104, 'Cybersecurity Expo', 'SecureNet', 'Exhibition', 680000, '2026-12-02'),
(105, 'Healthcare Technology Day', 'Care Group', 'Conference', 520000, '2026-11-18'),
(106, 'Startup Networking Night', 'Innovate Egypt', 'Networking', 300000, '2026-12-12')
""")


#### **EventAssignments**


In [ ]:
cursor.execute("""
INSERT INTO EventAssignments
(AssignmentID, StaffID, EventID, Role, HoursWorked, HourlyRate)
VALUES
(1, 1, 101, 'Event Coordinator', 120, 220),
(2, 3, 101, 'Technical Lead', 150, 320),
(3, 6, 101, 'Marketing Specialist', 100, 200),
(4, 4, 102, 'Operations Coordinator', 140, 230),
(5, 7, 102, 'Technical Consultant', 170, 350),
(6, 2, 102, 'Marketing Lead', 130, 280),
(7, 2, 103, 'Marketing Lead', 100, 280),
(8, 6, 103, 'Marketing Specialist', 120, 200),
(9, 9, 103, 'Marketing Assistant', 90, 160),
(10, 3, 104, 'Technical Lead', 160, 320),
(11, 5, 104, 'Security Specialist', 150, 300),
(12, 7, 104, 'Technical Consultant', 180, 350),
(13, 1, 105, 'Event Coordinator', 110, 220),
(14, 8, 105, 'Operations Manager', 145, 270),
(15, 10, 105, 'Operations Assistant', 100, 190),
(16, 4, 106, 'Operations Coordinator', 120, 230),
(17, 9, 106, 'Marketing Assistant', 110, 160),
(18, 5, 106, 'Technical Specialist', 90, 300),
(19, 8, 106, 'Operations Manager', 100, 270)
""")


### **Required Queries**
#### **Query 1 — JOIN + Calculated Column**

**Display:**

- Staff name 

- Event name 

- Event type 

- Role 

- Hours worked 

- Hourly rate 

- Total payment 

**Calculate:** TotalPayment = HoursWorked × HourlyRate

Sort the assignments from the highest total payment to the lowest.



In [ ]:
cursor.execute("""SELECT S.StaffName, 
E.EventName, 
E.EventType, 
A.Role,
A.HoursWorked,
A.HourlyRate,
(A.HoursWorked * A.HourlyRate) AS [Total payment]
FROM EventAssignments A JOIN Staff S
    ON S.StaffID = A.StaffID 
JOIN Events E
    ON A.EventID = E.EventID
ORDER BY [Total payment] DESC
""")
cursor.fetchall()


#### **Query 2 — JOIN + GROUP BY + Aggregate Functions**

**For each event, display:**

- Event name 

- Event type 

- Number of assigned staff members 

- Total hours worked 

- Average hourly rate 

- Total staff cost 

**Calculate:** TotalStaffCost = SUM(HoursWorked × HourlyRate)

Only display events whose total staff cost is greater than 90,000.
Sort by total staff cost descending.


In [ ]:
cursor.execute("""
SELECT E.EventName,
       E.EventType,
       COUNT(A.AssignmentID) AS [Number of assigned staff members],
       SUM(A.HoursWorked) AS [Total hours worked],
       AVG(A.HourlyRate) AS [Average hourly rate],
       SUM(A.HoursWorked * A.HourlyRate) AS [Total Staff Cost]
FROM Events E 
JOIN EventAssignments A
    ON E.EventID = A.EventID
GROUP BY E.EventName, E.EventID, E.EventType
HAVING [Total Staff Cost] > 90000
ORDER BY [Total Staff Cost] DESC
""")
cursor.fetchall()


#### **Query 3 — Three-Table JOIN + GROUP BY**

**For each department, display:**

- Department name 

- Number of different staff members who worked on events 

- Number of different events the department participated in 

- Total hours worked 

- Total amount earned by the department's staff 

Sort the departments by total amount earned descending.

**Important**

A staff member may participate in multiple events, so make sure the number of staff members does not count the same employee multiple times.


In [ ]:
cursor.execute("""
SELECT S.Department,
       COUNT(DISTINCT S.StaffID) AS [Number of different staff members],
       COUNT(DISTINCT E.EventID) AS [Number of different events],
       SUM(A.HoursWorked) AS [Total hours worked],
       SUM(A.HoursWorked * A.HourlyRate) AS [Total amount earned]
FROM EventAssignments A
JOIN Staff S
    ON S.StaffID = A.StaffID
JOIN Events E
    ON E.EventID = A.EventID
GROUP BY S.Department
ORDER BY [Total amount earned] DESC
""")
cursor.fetchall()


### **Query 4 — Correlated Subquery**

Find assignments where the employee's HourlyRate is greater than the average hourly rate of all staff working on the same event.

**Display:**

- Staff name 

- Event name 

- Hourly rate 

**For example:** you need to compare an employee's rate with Average HourlyRate for that specific EventID

**Requirement:** Use a correlated subquery.


In [ ]:
cursor.execute("""
SELECT S.StaffName,
       E.EventName,
       A.HourlyRate
FROM EventAssignments A
JOIN Staff S
    ON S.StaffID = A.StaffID
JOIN Events E
    ON E.EventID = A.EventID
WHERE A.HourlyRate > 
(SELECT AVG(HourlyRate) 
FROM EventAssignments
WHERE EventID = A.EventID
)
""")
cursor.fetchall()


### **Query 5 — Subquery + Aggregation**

Find the event or events with the highest total staff cost.

**Display:**

- Event ID 

- Event name 

- Event type 

- Total staff cost 

**Calculate:** TotalStaffCost = SUM(HoursWorked × HourlyRate)

You are not allowed to manually calculate or write the maximum cost.

Your SQL query must determine it automatically using a subquery.
If multiple events have the same maximum cost, return all of them.


In [ ]:
cursor.execute("""
SELECT E.EventID,
       E.EventName,
       E.EventType,
       SUM(A.HoursWorked * A.HourlyRate) AS [Total Staff Cost]
FROM Events E
JOIN EventAssignments A
    ON E.EventID = A.EventID
GROUP BY E.EventID,
       E.EventName,
       E.EventType
HAVING [Total Staff Cost] >= 
(SELECT MAX([Total Cost])
FROM
(SELECT SUM(HoursWorked * HourlyRate) AS [Total Cost]
FROM EventAssignments
GROUP BY EventID
) AS EventCosts)
""")
cursor.fetchall()


### **Query 6 — VIEW**

**Create a view named:** EventFinancialSummary

**The view should contain:**

- Event ID 

- Event name 

- Client name 

- Event type 

- Event budget 

- Number of assigned staff members 

- Total hours worked 

- Total staff cost 

- Remaining budget 

**Calculate:** RemainingBudget = Budget - TotalStaffCost

**After creating the view, retrieve only events whose:** RemainingBudget < 450000

Sort the result by RemainingBudget ascending.



In [ ]:
cursor.execute("""
CREATE VIEW EventFinancialSummary AS
SELECT E.EventID,
       E.EventName,
       E.ClientName,
       E.EventType,
       E.Budget,
       COUNT(DISTINCT S.StaffID) AS [Number of assigned staff members],
       SUM(A.HoursWorked) AS [Total hours worked],
       SUM(A.HoursWorked * A.HourlyRate) AS [Total Staff Cost],
       (E.Budget - SUM(A.HoursWorked * A.HourlyRate)) AS [Remaining budget]
FROM EventAssignments A
JOIN Staff S
    ON S.StaffID = A.StaffID
JOIN Events E
    ON E.EventID = A.EventID
GROUP BY E.EventID,
       E.EventName,
       E.ClientName,
       E.EventType,
       E.Budget
""")


In [ ]:
cursor.execute("""
SELECT *
FROM EventFinancialSummary
WHERE [Remaining budget] < 450000
ORDER BY [Remaining budget]
""")
cursor.fetchall()


### **Query 7 — CTE**

**Create a CTE named:** StaffWorkload

**The CTE should calculate for every staff member:**

- Staff ID 

- Staff name 

- Department 

- Number of events 

- Total hours worked 

- Total earnings from events 

**Where:** TotalEarnings = SUM(HoursWorked × HourlyRate)

**Then use the CTE to display only staff members who:** Worked on at least 2 events **AND** Worked more than 200 hours in total.

Sort by total hours descending.


In [ ]:
cursor.execute("""
WITH StaffWorkload AS
(
    SELECT S.StaffID,
           S.StaffName,
           S.Department,
           COUNT(E.EventID) AS [Number of events],
           SUM(A.HoursWorked) AS [Total hours worked],
           SUM(A.HoursWorked * A.HourlyRate) AS [Total earnings from events]
    FROM EventAssignments A
    JOIN Staff S
        ON S.StaffID = A.StaffID
    JOIN Events E
        ON E.EventID = A.EventID
    GROUP BY S.StaffID,
             S.StaffName,
             S.Department
)
SELECT *
FROM StaffWorkload
WHERE [Number of events] >= 2 
AND [Total hours worked] > 200
ORDER BY [Total hours worked] DESC
""")
cursor.fetchall()


### **Query 8 — Challenging: Two CTEs + Subquery**

#### **Use two CTEs.**

1. ##### **First CTE: StaffEventSummary**

    - **Calculate for every staff member:**
        - Staff ID 

        - Staff name 

        - Department 

        - Number of events 

        - Total hours worked 

        - Total earnings 

    **Where:** TotalEarnings = SUM(HoursWorked × HourlyRate)

2. ##### **Second CTE: DepartmentAverage**

    - **Calculate for every department:**
        - Department 

        - Average total earnings of staff in that department 

    - **Then display staff members whose:** TotalEarnings > Average TotalEarnings of their Department

##### **The final result should contain:**
- Department 

- Staff name 

- Number of events 

- Total hours worked 

- Total earnings 

- Department average earnings 

Sort first by department and then by total earnings descending.

##### **Requirements:**
- Use two CTEs. 
- Use aggregation. 
- Join the two CTEs.


In [ ]:
cursor.execute("""
WITH StaffEventSummary AS
(
    SELECT S.StaffID,
           S.StaffName,
           S.Department,
           COUNT(DISTINCT E.EventID) AS [Number of events],
           SUM(A.HoursWorked) AS [Total hours worked],
           SUM(A.HoursWorked * A.HourlyRate) AS [Total earnings]
    FROM EventAssignments A
    JOIN Staff S
        ON S.StaffID = A.StaffID
    JOIN Events E
        ON E.EventID = A.EventID
    GROUP BY S.StaffID,
           S.StaffName,
           S.Department
),
DepartmentAverage AS
(
    SELECT Department,
           AVG([Total earnings]) AS [Average total earnings]
    FROM StaffEventSummary
    GROUP BY Department
)
SELECT  S.StaffName,
        S.Department,
        S.[Number of events],
        S.[Total hours worked],
        S.[Total earnings],
        D.[Average total earnings]
FROM StaffEventSummary S
JOIN DepartmentAverage D
    ON S.Department = D.Department
WHERE S.[Total earnings] > D.[Average total earnings]
ORDER BY S.Department ASC,
         S.[Total earnings] DESC
""")
cursor.fetchall()
